# Cétolog : les données et ce qu'on attend de vous

Miraceti identifie les grands dauphins à leur nageoire dorsale. Nous recevons ses photos en zip, et l'objectif est de lui proposer les dauphins du catalogue qui ressemblent le plus à un individu photographié.

Les données passent par trois couches : `raw/` (les zips, intouchables), `curated/` (les données prêtes) et `derived/` (ce qui est calculé). Leur origine et leurs pièges sont décrits dans `docs/miraceti_data/catalogue.md`.

**La qualité de ce que vous livrez est votre responsabilité** : à vous de définir vos contrôles et de montrer dans votre PR que la livraison est fiable.

**Avant de commencer :** demandez vos accès (`docs/miraceti_data/acces.md`) installez l'environnement (`uv sync --all-groups`, voir le README), puis choisissez le kernel `.venv` dans le notebook.

**Deux règles :** on ne modifie jamais `raw/` ni le bucket, et on efface les sorties du notebook avant de commiter, car les photos de Miraceti ne sont pas publiques.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # .env et data/ sont lus depuis la racine du dépôt
sys.path.insert(0, str(ROOT))
from pipeline.storage import connect  # noqa: E402

bucket = connect()
bucket.check()

## Data engineering

**Vous partez des zips de Miraceti** : l'historique, le catalogue et les best-of.

**On veut :**
- des photos rangées par jeu, livraison et individu, qui gardent leur nom d'origine ;
- pour chaque livraison, un manifeste avec une ligne par photo : dauphin, côté, dates, taille et empreinte du fichier ;
- les anomalies tracées, jamais corrigées : seul Miraceti peut trancher ;
- une chaîne qu'on peut relancer sans risque, et qui accepte une nouvelle livraison sans réécriture ;
- une donnée dans laquelle le ML peut avoir confiance.

In [ ]:
bucket.pull(prefix="raw/")  # les zips : environ 3,5 Go

## Analytics engineering

**Vous partez des manifestes et des propositions** de la version actuelle.

**On veut :**
- une base de données reconstructible, à la place des fichiers ;
- le lien entre un individu provisoire et un dauphin du catalogue, avec les décisions de Miraceti, qui n'ont aucune place aujourd'hui ;
- un historique des versions du catalogue ;
- avec Miraceti, une couche gold : suivi de chaque dauphin, carte des observations, statistiques, en explorant leur projet QGIS.

In [ ]:
for kind in ("historique", "catalogue", "best_of"):
    bucket.pull(prefix=f"curated/{kind}/manifests/")
#bucket.pull(prefix="derived/propositions/")

## ML et MLOps

**Vous partez des empreintes CetaceaNet**, et des photos prêtes si vous en avez besoin.

**On veut :**
- savoir ce que vaut vraiment le modèle actuel, sans qu'il se contente de retrouver sa propre photo ;
- comparer d'autres modèles sur une base équitable (pistes dans `docs/cetolog_ml/a_tester.md`) ;
- repérer les dauphins qui ne sont pas encore au catalogue ;
- des résultats reproductibles et comparables d'une version à l'autre ;
- une façon sûre de mettre une nouvelle version en production, et de revenir en arrière.

In [ ]:
bucket.pull(prefix="derived/embeddings/")  # quelques Mo
# bucket.pull(prefix="curated/")  # les photos prêtes : environ 3,4 Go